In [163]:
import pandas as pd
pd.set_option('display.max_rows', None)

In [164]:
TERMINALS = (
    'if',
    ',',
    'foreach',
    'equal',
    '=',
    '>',
    '<',
    '&',
    '|',
    '!',
    '-',
    '+',
    '/',
    '*',
    'resolve',
    'FUNCTION_NAME',
    '(',
    ')',
    # 'eps',
    'NUMERIC',
    'BOOL',
)

In [165]:
EPSILON = 'eps'

In [166]:
NOTTERMINALS_FOLLOW_SYMBOLS = {
    'S': ['FUNCTION_NAME', 'NUMERIC', 'BOOL', 'if', 'foreach', 'equal', 'resolve', '+', '-', '=', '>', '<', '|', '&', '!', '/', '*'],
    'EXPRESSION': ['FUNCTION_NAME', 'NUMERIC', 'BOOL', 'if', 'foreach', 'equal', 'resolve', '+', '-', '=', '>', '<', '|', '&', '!', '/', '*'],
    'ARG': ['NUMERIC', 'BOOL'],
    'TRIPLET_CALL': ['(', 'eps'],
    'DOUBLE_CALL': ['(', 'eps'],
    'SINGLE_CALL': ['(', 'eps'],
    'RESOLVE_CALL': ['(', 'eps'],
    'CALL': ['(', 'eps'],
    'EXPRESSIONS': ['FUNCTION_NAME', 'NUMERIC', 'BOOL', 'if', 'foreach', 'equal', 'resolve', '+', '-', '=', '>', '<', '|', '&', '!', '/', '*', 'eps', ')', ','],
    'LIST_OF_EXPRESSIONS': ['eps', ',', ')'],
    'eps': [',', ')']
}

In [167]:
RULES = """
S ::= EXPRESSION
EXPRESSION ::= ARG
EXPRESSION ::= if TRIPLET_CALL
EXPRESSION ::= foreach DOUBLE_CALL
EXPRESSION ::= equal DOUBLE_CALL
EXPRESSION ::= = DOUBLE_CALL
EXPRESSION ::= > DOUBLE_CALL
EXPRESSION ::= < DOUBLE_CALL
EXPRESSION ::= & DOUBLE_CALL
EXPRESSION ::= | DOUBLE_CALL
EXPRESSION ::= ! SINGLE_CALL
EXPRESSION ::= - DOUBLE_CALL
EXPRESSION ::= + DOUBLE_CALL
EXPRESSION ::= / DOUBLE_CALL
EXPRESSION ::= * DOUBLE_CALL
EXPRESSION ::= resolve RESOLVE_CALL
EXPRESSION ::= FUNCTION_NAME CALL
SINGLE_CALL ::= ( EXPRESSION )
SINGLE_CALL ::= eps
DOUBLE_CALL ::= ( EXPRESSION , EXPRESSION )
DOUBLE_CALL ::= eps
TRIPLET_CALL ::= ( EXPRESSION , EXPRESSION , EXPRESSION )
TRIPLET_CALL ::= eps
RESOLVE_CALL ::= ( FUNCTION_NAME , EXPRESSION )
RESOLVE_CALL ::= eps
CALL ::= ( EXPRESSIONS
CALL ::= eps
EXPRESSIONS ::= )
EXPRESSIONS ::= EXPRESSION LIST_OF_EXPRESSIONS
LIST_OF_EXPRESSIONS ::= )
LIST_OF_EXPRESSIONS ::= , EXPRESSION LIST_OF_EXPRESSIONS
ARG ::= NUMERIC
ARG ::= BOOL
"""[1:-1]

In [168]:
separator = ' ::= '
right_separator = ' '

In [169]:
def left_right_delimeter(r):
    return r.split(separator)


def right_delimeter(rr):
    return rr.split(right_separator)


In [170]:
data = dict()
IDX = 1
for r in RULES.split('\n'):
    lr, rr = r.split(separator)
    data[IDX] = {lr: rr.split(right_separator)}
    IDX += 1


In [171]:
for idx, r in data.items():
    d = dict()
    lr, rr = next(iter(r.items()))
    data[idx] = {lr: {i + IDX: v for i, v in enumerate(rr)}}
    IDX += len(rr)
    

In [172]:
print(str(data)[1:-1].replace('}}, ', '\n'))

1: {'S': {34: 'EXPRESSION'
2: {'EXPRESSION': {35: 'ARG'
3: {'EXPRESSION': {36: 'if', 37: 'TRIPLET_CALL'
4: {'EXPRESSION': {38: 'foreach', 39: 'DOUBLE_CALL'
5: {'EXPRESSION': {40: 'equal', 41: 'DOUBLE_CALL'
6: {'EXPRESSION': {42: '=', 43: 'DOUBLE_CALL'
7: {'EXPRESSION': {44: '>', 45: 'DOUBLE_CALL'
8: {'EXPRESSION': {46: '<', 47: 'DOUBLE_CALL'
9: {'EXPRESSION': {48: '&', 49: 'DOUBLE_CALL'
10: {'EXPRESSION': {50: '|', 51: 'DOUBLE_CALL'
11: {'EXPRESSION': {52: '!', 53: 'SINGLE_CALL'
12: {'EXPRESSION': {54: '-', 55: 'DOUBLE_CALL'
13: {'EXPRESSION': {56: '+', 57: 'DOUBLE_CALL'
14: {'EXPRESSION': {58: '/', 59: 'DOUBLE_CALL'
15: {'EXPRESSION': {60: '*', 61: 'DOUBLE_CALL'
16: {'EXPRESSION': {62: 'resolve', 63: 'RESOLVE_CALL'
17: {'EXPRESSION': {64: 'FUNCTION_NAME', 65: 'CALL'
18: {'SINGLE_CALL': {66: '(', 67: 'EXPRESSION', 68: ')'
19: {'SINGLE_CALL': {69: 'eps'
20: {'DOUBLE_CALL': {70: '(', 71: 'EXPRESSION', 72: ',', 73: 'EXPRESSION', 74: ')'
21: {'DOUBLE_CALL': {75: 'eps'
22: {'TRIPLET_CALL': 

In [173]:
table = []

In [174]:
for idx, r in data.items():
    next_value = next(iter(next(iter(r.values())).keys()))
    current_token, current_token_value = next(iter(r.items()))
    current_token_value = current_token_value[next_value]
    if idx != len(data):
        next_token = next(iter(data[idx + 1].keys()))
        v = next_token != current_token
    else:
        v = True
    table.append({'FROM': idx, 'FOLLOW_SYMBOLS': current_token_value, 'NEXT': next_value, 'ACCEPT': False, 'STACK': False, 'ERROR': v, 'RETURN': False})

In [175]:
for r in data.values():
    k = 0
    ri = next(iter(r.values()))
    l = len(ri)
    for a, b in ri.items():
        t = None
        k += 1
        if b not in TERMINALS:
            for idx_k, v_k in data.items():
                d = next(iter(v_k.keys()))
                if d == b:
                    t = idx_k
                    break
            pr = {'FROM': a, 'FOLLOW_SYMBOLS': b, 'NEXT': t, 'ACCEPT': b in TERMINALS, 'STACK': (b not in TERMINALS) and (k != l), 'ERROR': True, 'RETURN': (k == l) and ((b == EPSILON) or (b in TERMINALS))}
        else:
            pr = {'FROM': a, 'FOLLOW_SYMBOLS': b, 'NEXT': t if len(ri) == 1 else a + 1, 'ACCEPT': b in TERMINALS, 'STACK': (b not in TERMINALS) and (k != l), 'ERROR': True, 'RETURN': (k == l) and ((b == EPSILON) or (b in TERMINALS))}
        table.append(pr)


In [176]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [177]:
tpd = pd.DataFrame(table)
for i in ['FROM', 'NEXT', 'ACCEPT', 'STACK', 'ERROR', 'RETURN']:
    r = tpd[i].isna().map(lambda a: not a)
    tpd.loc[r, i] = tpd.loc[r, i].astype('uint8')
tpd = tpd.set_index('FROM', drop=True)
tpd['FOLLOW_SYMBOLS'] = tpd['FOLLOW_SYMBOLS'].apply(lambda a: ' '.join(NOTTERMINALS_FOLLOW_SYMBOLS[a]) if a in NOTTERMINALS_FOLLOW_SYMBOLS else a)
tpd.to_csv('./lexer_table.csv')
tpd

,FOLLOW_SYMBOLS,NEXT,ACCEPT,STACK,ERROR,RETURN
FROM,,,,,,
1,FUNCTION_NAME NUMERIC BOOL if foreach equal re...,34.0,0,0,1,0
2,NUMERIC BOOL,35.0,0,0,0,0
3,if,36.0,0,0,0,0
4,foreach,38.0,0,0,0,0
5,equal,40.0,0,0,0,0
6,=,42.0,0,0,0,0
7,>,44.0,0,0,0,0
8,<,46.0,0,0,0,0
9,&,48.0,0,0,0,0
